# LIMPIEZA DE DATOS — PARADOS (EPA)
### Encuesta de Población Activa: series de desempleo por sexo y edad, 1976–2026
### Incluye:
- Importación de librerías
- Carga de datasets
- Conversión de tipos y limpieza
- Unión de series y homogenización metodológica
- Exportación de archivos finales

---
**Fuentes (INE, Encuesta de Población Activa):**
- `epa_corregida_76_04.xls`, *Parados por sexo y grupo de edad (serie corregida).* Cobertura: 1976T3–2004T4. Trimestral.
- `Parados por sexo y grupo de edad. Valores absolutos y porcentajes respecto del total de cada sexo.csv`, *Parados por sexo y grupo de edad.* Cobertura: 2002T1–2026T1. Trimestral.

# 0. Importación de "Librerías"

In [75]:
import numpy as np
import pandas as pd
from functools import reduce
import plotly.express as px

# 1. Carga de datasets

In [76]:
ruta= '../Fuentes/INE/EPA/0. Objetivo/'

### Serie de Parados de la EPA de 1976 a 2004, extrapolada hacia detrás debido a cambios de metodología en 2001

In [77]:
df_76_04 = pd.ExcelFile(ruta + 'epa_corregida_76_04.xls')
df_76_04
print('Hojas disponibles:', df_76_04.sheet_names)

Hojas disponibles: ['Tabla AMBOS SEXOS', 'Gráficos AMBOS SEXOS', 'Tabla VARONES', 'Gráficos VARONES', 'Tabla MUJERES', 'Gráficos MUJERES']


### Tabla para ambos sexos (total)

In [78]:
df_ambos_sexos = df_76_04.parse('Tabla AMBOS SEXOS', header=3)
print('Dimensiones Tabla AMBOS SEXOS:', df_ambos_sexos.shape)
df_ambos_sexos

Dimensiones Tabla AMBOS SEXOS: (114, 11)


,año,trimestre,parados_total,diferencia_parados_total,parados_menos_de_25,diferencia_parados_menos_de_25,parados_25_o_mas,diferencia_parados_25_o_mas,parados_total_antigua,parados_menos_de_25_antigua,parados_25_o_mas_antigua
0,1976,3,589.000000,0.000000,280.000000,0.000000,309.000000,0.000000,589.00000,280.00000,309.00000
1,1976,4,627.771632,-0.228368,303.871632,-0.128368,323.900000,0.000000,628.00000,304.00000,323.90000
2,1977,1,652.867832,-1.232168,313.474892,-0.625108,339.392940,-0.507060,654.10000,314.10000,339.90000
3,1977,2,629.136744,-2.263256,307.356591,-0.943409,321.780153,-1.319847,631.40000,308.30000,323.10000
4,1977,3,707.184698,-3.715302,373.656013,-1.943987,333.528686,-1.871314,710.90000,375.60000,335.40000
...,...,...,...,...,...,...,...,...,...,...,...
109,2003,4,2252.800000,-408.909310,552.200000,-75.452270,1700.600000,-333.457040,2661.70931,627.65227,2034.05704
110,2004,1,2287.300000,-390.647440,540.000000,-74.674450,1747.300000,-315.972990,2677.94744,614.67445,2063.27299
111,2004,2,2227.700000,-365.899550,536.400000,-65.418940,1691.300000,-300.480610,2593.59955,601.81894,1991.78061
112,2004,3,2181.900000,-381.754350,538.900000,-67.236190,1643.000000,-314.518160,2563.65435,606.13619,1957.51816


In [79]:
df_ambos_sexos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114 entries, 0 to 113
Data columns (total 11 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   año                             114 non-null    int64  
 1   trimestre                       114 non-null    int64  
 2   parados_total                   114 non-null    float64
 3   diferencia_parados_total        114 non-null    float64
 4   parados_menos_de_25             114 non-null    float64
 5   diferencia_parados_menos_de_25  114 non-null    float64
 6   parados_25_o_mas                114 non-null    float64
 7   diferencia_parados_25_o_mas     114 non-null    float64
 8   parados_total_antigua           114 non-null    float64
 9   parados_menos_de_25_antigua     114 non-null    float64
 10  parados_25_o_mas_antigua        114 non-null    float64
dtypes: float64(9), int64(2)
memory usage: 9.9 KB


### Tabla para hombres

In [80]:
df_hombres = df_76_04.parse('Tabla VARONES', header=3)
print('Dimensiones Tabla VARONES:', df_hombres.shape)
df_hombres

Dimensiones Tabla VARONES: (114, 11)


,año,trimestre,parados_varones_total,diferencia_parados_varones_total,parados_varones_menos_de_25,diferencia_parados_varones_menos_de_25,parados_varones_25_o_mas,diferencia_parados_varones_25_o_mas,parados_varones_total_antigua,parados_varones_menos_de_25_antigua,parados_varones_25_o_mas_antigua
0,1976,3,406.900000,0.000000,151.100000,0.000000,255.800000,0.000000,406.90000,151.10000,255.80000
1,1976,4,441.214792,-0.185208,166.871632,-0.128368,274.343160,-0.056840,441.40000,167.00000,274.40000
2,1977,1,464.104722,-1.095278,174.703159,-0.596841,289.401563,-0.498437,465.20000,175.30000,289.90000
3,1977,2,445.055802,-1.744198,170.481873,-0.618127,274.573929,-1.126071,446.80000,171.10000,275.70000
4,1977,3,483.406415,-2.193585,202.764550,-0.835450,280.641865,-1.458135,485.60000,203.60000,282.10000
...,...,...,...,...,...,...,...,...,...,...,...
109,2003,4,993.900000,-148.223920,269.900000,-36.567760,724.000000,-111.656160,1142.12392,306.46776,835.65616
110,2004,1,1008.600000,-141.883470,264.200000,-36.722150,744.400000,-105.161320,1150.48347,300.92215,849.56132
111,2004,2,972.500000,-135.957060,262.100000,-31.160260,710.400000,-104.796800,1108.45706,293.26026,815.19680
112,2004,3,970.900000,-133.219040,260.600000,-28.799980,710.300000,-104.419060,1104.11904,289.39998,814.71906


In [81]:
df_hombres.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114 entries, 0 to 113
Data columns (total 11 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   año                                     114 non-null    int64  
 1   trimestre                               114 non-null    int64  
 2   parados_varones_total                   114 non-null    float64
 3   diferencia_parados_varones_total        114 non-null    float64
 4   parados_varones_menos_de_25             114 non-null    float64
 5   diferencia_parados_varones_menos_de_25  114 non-null    float64
 6   parados_varones_25_o_mas                114 non-null    float64
 7   diferencia_parados_varones_25_o_mas     114 non-null    float64
 8   parados_varones_total_antigua           114 non-null    float64
 9   parados_varones_menos_de_25_antigua     114 non-null    float64
 10  parados_varones_25_o_mas_antigua        114 non-null    float6

### Tabla para mujeres

In [82]:
df_mujeres = df_76_04.parse('Tabla MUJERES', header=3)
print('Dimensiones Tabla MUJERES:', df_mujeres.shape)
df_mujeres

Dimensiones Tabla MUJERES: (114, 11)


,año,trimestre,parados_mujeres_total,diferencia_parados_mujeres_total,parados_mujeres_menos_de_25,diferencia_parados_mujeres_menos_de_25,parados_mujeres_25_o_mas,diferencia_parados_mujeres_25_o_mas,parados_mujeres_total_antigua,parados_mujeres_menos_de_25_antigua,parados_mujeres_25_o_mas_antigua
0,1976,3,182.100000,0.000000,129.000000,0.000000,53.200000,0.000000,182.10000,129.00000,53.20000
1,1976,4,186.600000,0.000000,137.000000,0.000000,49.698869,-0.001131,186.60000,137.00000,49.70000
2,1977,1,188.763110,-0.136890,138.771733,-0.228267,49.991377,-0.008623,188.90000,139.00000,50.00000
3,1977,2,184.080942,-0.519058,136.874718,-0.425282,47.206224,-0.193776,184.60000,137.30000,47.40000
4,1977,3,223.778283,-1.421717,170.891463,-1.108537,52.886821,-0.313179,225.20000,172.00000,53.20000
...,...,...,...,...,...,...,...,...,...,...,...
109,2003,4,1258.900000,-260.685390,282.300000,-38.884510,976.600000,-221.800880,1519.58539,321.18451,1198.40088
110,2004,1,1278.700000,-248.763970,275.800000,-37.952300,1002.900000,-210.811670,1527.46397,313.75230,1213.71167
111,2004,2,1255.200000,-229.942490,274.300000,-34.258680,980.900000,-195.683810,1485.14249,308.55868,1176.58381
112,2004,3,1211.000000,-248.535310,278.300000,-38.436210,932.700000,-210.099100,1459.53531,316.73621,1142.79910


In [83]:
df_mujeres.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114 entries, 0 to 113
Data columns (total 11 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   año                                     114 non-null    int64  
 1   trimestre                               114 non-null    int64  
 2   parados_mujeres_total                   114 non-null    float64
 3   diferencia_parados_mujeres_total        114 non-null    float64
 4   parados_mujeres_menos_de_25             114 non-null    float64
 5   diferencia_parados_mujeres_menos_de_25  114 non-null    float64
 6   parados_mujeres_25_o_mas                114 non-null    float64
 7   diferencia_parados_mujeres_25_o_mas     114 non-null    float64
 8   parados_mujeres_total_antigua           114 non-null    float64
 9   parados_mujeres_menos_de_25_antigua     114 non-null    float64
 10  parados_mujeres_25_o_mas_antigua        114 non-null    float6

## Unión en un único dataset corregido

In [84]:
dataframes = [df_ambos_sexos, df_hombres, df_mujeres]

for i, df in enumerate(dataframes):
    texto_fecha = df['año'].astype('string') + "Q" + df['trimestre'].astype('string')
    df['periodo'] = texto_fecha
    df['fecha'] = pd.PeriodIndex(texto_fecha, freq="Q").to_timestamp()
    dataframes[i] = df

df_76_04 = reduce(lambda left, right: pd.merge(left, right, on='periodo', how='outer'), dataframes)
df_76_04 = df_76_04.drop(columns= ['fecha_x', 'fecha_y', 'año_x', 'año_y', 'trimestre_x', 'trimestre_y'])

#Reordenando columnas
columnas_temporales = ['fecha', 'año', 'trimestre', 'periodo']
columnas_valores = [columna for columna in df_76_04.columns if columna not in columnas_temporales]
df_76_04 = df_76_04[columnas_temporales + columnas_valores]
df_76_04

,fecha,año,trimestre,periodo,parados_total,diferencia_parados_total,parados_menos_de_25,diferencia_parados_menos_de_25,parados_25_o_mas,diferencia_parados_25_o_mas,...,parados_varones_25_o_mas_antigua,parados_mujeres_total,diferencia_parados_mujeres_total,parados_mujeres_menos_de_25,diferencia_parados_mujeres_menos_de_25,parados_mujeres_25_o_mas,diferencia_parados_mujeres_25_o_mas,parados_mujeres_total_antigua,parados_mujeres_menos_de_25_antigua,parados_mujeres_25_o_mas_antigua
0,1976-07-01,1976,3,1976Q3,589.000000,0.000000,280.000000,0.000000,309.000000,0.000000,...,255.80000,182.100000,0.000000,129.000000,0.000000,53.200000,0.000000,182.10000,129.00000,53.20000
1,1976-10-01,1976,4,1976Q4,627.771632,-0.228368,303.871632,-0.128368,323.900000,0.000000,...,274.40000,186.600000,0.000000,137.000000,0.000000,49.698869,-0.001131,186.60000,137.00000,49.70000
2,1977-01-01,1977,1,1977Q1,652.867832,-1.232168,313.474892,-0.625108,339.392940,-0.507060,...,289.90000,188.763110,-0.136890,138.771733,-0.228267,49.991377,-0.008623,188.90000,139.00000,50.00000
3,1977-04-01,1977,2,1977Q2,629.136744,-2.263256,307.356591,-0.943409,321.780153,-1.319847,...,275.70000,184.080942,-0.519058,136.874718,-0.425282,47.206224,-0.193776,184.60000,137.30000,47.40000
4,1977-07-01,1977,3,1977Q3,707.184698,-3.715302,373.656013,-1.943987,333.528686,-1.871314,...,282.10000,223.778283,-1.421717,170.891463,-1.108537,52.886821,-0.313179,225.20000,172.00000,53.20000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,2003-10-01,2003,4,2003Q4,2252.800000,-408.909310,552.200000,-75.452270,1700.600000,-333.457040,...,835.65616,1258.900000,-260.685390,282.300000,-38.884510,976.600000,-221.800880,1519.58539,321.18451,1198.40088
110,2004-01-01,2004,1,2004Q1,2287.300000,-390.647440,540.000000,-74.674450,1747.300000,-315.972990,...,849.56132,1278.700000,-248.763970,275.800000,-37.952300,1002.900000,-210.811670,1527.46397,313.75230,1213.71167
111,2004-04-01,2004,2,2004Q2,2227.700000,-365.899550,536.400000,-65.418940,1691.300000,-300.480610,...,815.19680,1255.200000,-229.942490,274.300000,-34.258680,980.900000,-195.683810,1485.14249,308.55868,1176.58381
112,2004-07-01,2004,3,2004Q3,2181.900000,-381.754350,538.900000,-67.236190,1643.000000,-314.518160,...,814.71906,1211.000000,-248.535310,278.300000,-38.436210,932.700000,-210.099100,1459.53531,316.73621,1142.79910


In [85]:
df_76_04.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114 entries, 0 to 113
Data columns (total 31 columns):
 #   Column                                  Non-Null Count  Dtype         
---  ------                                  --------------  -----         
 0   fecha                                   114 non-null    datetime64[ns]
 1   año                                     114 non-null    int64         
 2   trimestre                               114 non-null    int64         
 3   periodo                                 114 non-null    string        
 4   parados_total                           114 non-null    float64       
 5   diferencia_parados_total                114 non-null    float64       
 6   parados_menos_de_25                     114 non-null    float64       
 7   diferencia_parados_menos_de_25          114 non-null    float64       
 8   parados_25_o_mas                        114 non-null    float64       
 9   diferencia_parados_25_o_mas             114 non-null  

### Serie de 2002 a 2025

In [86]:
df_02_25 = pd.read_csv(ruta + 'Parados por sexo y grupo de edad. Valores absolutos y porcentajes respecto del total de cada sexo.csv', sep=';', encoding='utf-8-sig')
df_02_25

,Edad,Sexo,Unidad,Periodo,Total
0,Total,Ambos sexos,Valor absoluto,2025T4,"2.477,1"
1,Total,Ambos sexos,Valor absoluto,2025T3,"2.613,2"
2,Total,Ambos sexos,Valor absoluto,2025T2,"2.553,1"
3,Total,Ambos sexos,Valor absoluto,2025T1,"2.789,2"
4,Total,Ambos sexos,Valor absoluto,2024T4,"2.595,5"
...,...,...,...,...,...
7483,70 y más años,Mujeres,Porcentaje,2003T1,0
7484,70 y más años,Mujeres,Porcentaje,2002T4,0
7485,70 y más años,Mujeres,Porcentaje,2002T3,0
7486,70 y más años,Mujeres,Porcentaje,2002T2,0


In [87]:
df_02_25 = df_02_25[df_02_25['Unidad'] != 'Porcentaje'].drop(columns=['Unidad'])
df_02_25.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3744 entries, 0 to 7391
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Edad     3744 non-null   object
 1   Sexo     3744 non-null   object
 2   Periodo  3744 non-null   object
 3   Total    3744 non-null   object
dtypes: object(4)
memory usage: 146.2+ KB


### Chequeo en conjunto

In [88]:
dataframes = [df_76_04, df_02_25]

df_76_04.attrs['nombre'] = 'df_76_04'
df_02_25.attrs['nombre'] = 'df_02_25'

In [89]:
# Valores únicos de cada dataset para ver las categorías
for df in dataframes:
    print()
    print(f'Valores únicos en {df.attrs['nombre']}')
    for columna in df:
        print(df[columna].unique())
        print()


Valores únicos en df_76_04
<DatetimeArray>
['1976-07-01 00:00:00', '1976-10-01 00:00:00', '1977-01-01 00:00:00',
 '1977-04-01 00:00:00', '1977-07-01 00:00:00', '1977-10-01 00:00:00',
 '1978-01-01 00:00:00', '1978-04-01 00:00:00', '1978-07-01 00:00:00',
 '1978-10-01 00:00:00',
 ...
 '2002-07-01 00:00:00', '2002-10-01 00:00:00', '2003-01-01 00:00:00',
 '2003-04-01 00:00:00', '2003-07-01 00:00:00', '2003-10-01 00:00:00',
 '2004-01-01 00:00:00', '2004-04-01 00:00:00', '2004-07-01 00:00:00',
 '2004-10-01 00:00:00']
Length: 114, dtype: datetime64[ns]

[1976 1977 1978 1979 1980 1981 1982 1983 1984 1985 1986 1987 1988 1989
 1990 1991 1992 1993 1994 1995 1996 1997 1998 1999 2000 2001 2002 2003
 2004]

[3 4 1 2]

<StringArray>
['1976Q3', '1976Q4', '1977Q1', '1977Q2', '1977Q3', '1977Q4', '1978Q1',
 '1978Q2', '1978Q3', '1978Q4',
 ...
 '2002Q3', '2002Q4', '2003Q1', '2003Q2', '2003Q3', '2003Q4', '2004Q1',
 '2004Q2', '2004Q3', '2004Q4']
Length: 114, dtype: string

[ 589.          627.77163198  652.8

In [90]:
# Información acerca del tipo de dato y valores nulos de las columnas de cada dataset
for df in dataframes:
    print(f'Información acerca de {df.attrs['nombre']}')
    print(df.info())

Información acerca de df_76_04
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114 entries, 0 to 113
Data columns (total 31 columns):
 #   Column                                  Non-Null Count  Dtype         
---  ------                                  --------------  -----         
 0   fecha                                   114 non-null    datetime64[ns]
 1   año                                     114 non-null    int64         
 2   trimestre                               114 non-null    int64         
 3   periodo                                 114 non-null    string        
 4   parados_total                           114 non-null    float64       
 5   diferencia_parados_total                114 non-null    float64       
 6   parados_menos_de_25                     114 non-null    float64       
 7   diferencia_parados_menos_de_25          114 non-null    float64       
 8   parados_25_o_mas                        114 non-null    float64       
 9   diferencia_parados_25_o

# 2. Limpieza

### Transformación del tipo de dato y limpieza inicial

In [91]:
# Conversión de las variables categóricas a "category" por eficiencia
variables_categoricas = ['Sexo', 'Edad', 'Unidad']

for i, df in enumerate(dataframes):
    for variable in variables_categoricas:
        if variable in df.columns:
            df[variable] = df[variable].astype('category')
        else:
            continue
    dataframes[i] = df

### Conversión de los datos numéricos a formato internacional, eliminar puntos y reemplazar las comas "," por puntos "."

In [92]:
columnas_temporales += ['Periodo']


columnas_valores = [columna for columna in df_02_25.columns if columna not in columnas_temporales and columna not in variables_categoricas]
for columna in columnas_valores:
    # Convirtiendo a texto para poder manipularlo
    valores_limpios = df_02_25[columna].astype(str).str.strip()

    # Eliminando los espacios internos y espacios irrompibles (\xa0) que usa el INE
    valores_limpios = valores_limpios.str.replace(' ', '', regex=False)
    valores_limpios = valores_limpios.str.replace('\xa0', '', regex=False)

    # Eliminando el punto de miles y cambiando la coma por punto
    valores_limpios = valores_limpios.str.replace('.', '', regex=False).str.replace(',', '.', regex=False)

    # Convirtiendo a numérico a numérico
    df_02_25[columna] = pd.to_numeric(valores_limpios, errors='coerce')

    # Rellenando los nulos con 0.0 (valores que el INE guardaba con '..'
    df_02_25[columna] = df_02_25[columna].fillna(np.nan)

# Actualizamos la lista
dataframes[1] = df_02_25

In [93]:
# Información acerca del tipo de dato y valores nulos de las columnas de cada dataset
for df in dataframes:
    print(f'Información acerca de {df.attrs['nombre']}')
    print(df.info())

Información acerca de df_76_04
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114 entries, 0 to 113
Data columns (total 31 columns):
 #   Column                                  Non-Null Count  Dtype         
---  ------                                  --------------  -----         
 0   fecha                                   114 non-null    datetime64[ns]
 1   año                                     114 non-null    int64         
 2   trimestre                               114 non-null    int64         
 3   periodo                                 114 non-null    string        
 4   parados_total                           114 non-null    float64       
 5   diferencia_parados_total                114 non-null    float64       
 6   parados_menos_de_25                     114 non-null    float64       
 7   diferencia_parados_menos_de_25          114 non-null    float64       
 8   parados_25_o_mas                        114 non-null    float64       
 9   diferencia_parados_25_o

### Transformación de los rangos de edad en las series temporales más recientes (2002 - 2025) para hgomogeneizar el dataset

In [94]:
categorias_mas_25 = [
    'De 25 a 29 años',
    'De 30 a 34 años',
    'De 35 a 39 años',
    'De 40 a 44 años',
    'De 45 a 49 años',
    'De 50 a 54 años',
    'De 55 a 59 años',
    'De 60 a 64 años',
    'De 65 a 69 años',
    '70 y más años'
]


categorias_menos_25 = [
    'De 16 a 19 años',
    'De 20 a 24 años'
]

df_02_25['Edad'] = df_02_25['Edad'].astype(str).str.strip()

columnas_agrupacion = ['Sexo', 'Periodo']
for col in columnas_agrupacion:
    df_02_25[col] = df_02_25[col].astype(str)

# Creando la máscara para filtrar
mask_mas = df_02_25['Edad'].isin(categorias_mas_25)
mask_menos = df_02_25['Edad'].isin(categorias_menos_25)

# Agrupando los datos filtrados y sumamos el 'Total'
df_mas = df_02_25[mask_mas].groupby(columnas_agrupacion, as_index=False, observed=True)['Total'].sum(min_count=1)
df_menos = df_02_25[mask_menos].groupby(columnas_agrupacion, as_index=False, observed=True)['Total'].sum(min_count=1)

# Creando la columna 'edad' con el nombre unificado
df_mas['Edad'] = '25_o_mas'
df_menos['Edad'] = 'menos_de_25'

# Eliminando las categorías no agrupadas
df_02_25 = df_02_25[~(mask_mas | mask_menos)]

# Unimos la nueva categoría al dataset original del que provenían los datos
df_02_25 = pd.concat([df_02_25, df_mas, df_menos], ignore_index=True)
df_02_25['Edad'] = df_02_25['Edad'].astype(str).str.lower()
df_02_25['Edad'] = df_02_25['Edad'].astype('category')

for col in columnas_agrupacion:
    df_02_25[col] = df_02_25[col].astype('category')

dataframes[1] = df_02_25

In [95]:
print(df_02_25['Edad'].unique())

['total', '25_o_mas', 'menos_de_25']
Categories (3, object): ['25_o_mas', 'menos_de_25', 'total']


### Transformación de los datos de "Periodo"

In [96]:
trimestres_map = {
    'I': '1',
    'II': '2',
    'III': '3',
    'IV': '4',
    '1': '1', '2': '2', '3': '3', '4': '4'
}

# Creación de la variable periodo en la serie antigua
texto_fecha = df_76_04['año'].astype('string') + "Q" + df_76_04['trimestre'].astype('string')
df_76_04['periodo'] = texto_fecha.astype('category')

# Conversión de "Periodo" a string y a formato internacional para manejarlo mejor
df_02_25['Periodo'] = df_02_25['Periodo'].astype('string')

# Separación de la información en diferentes columnas para su posterior uso
df_02_25['año'] = df_02_25['Periodo'].str[:4]
df_02_25['trimestre'] = df_02_25['Periodo'].str[5:].map(trimestres_map)

texto_fecha = df_02_25['año'].astype('string') + "Q" + df_02_25['trimestre'].astype('string')
df_02_25['Periodo'] = texto_fecha.astype('category')
df_02_25['fecha'] = pd.PeriodIndex(texto_fecha, freq="Q").to_timestamp()

df_02_25['año'] = df_02_25['año'].astype(int)
df_02_25['trimestre'] = df_02_25['trimestre'].astype(int)
df_02_25 = df_02_25.rename(columns= {'Periodo': 'periodo'})
dataframes = [df_76_04, df_02_25]

for i, df in enumerate(dataframes):
    columnas_temporales = ['fecha', 'año', 'trimestre', 'periodo']
    columnas_valores = [columna for columna in df.columns if columna not in columnas_temporales]
    df = df[columnas_temporales + columnas_valores]
    df = df.set_index('fecha').sort_index()
    df = df.reset_index()
    dataframes[i] = df

    print(f'Información acerca de {df.attrs['nombre']}')
    print(df.info())

df_76_04, df_02_25 = dataframes

Información acerca de df_76_04
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114 entries, 0 to 113
Data columns (total 31 columns):
 #   Column                                  Non-Null Count  Dtype         
---  ------                                  --------------  -----         
 0   fecha                                   114 non-null    datetime64[ns]
 1   año                                     114 non-null    int64         
 2   trimestre                               114 non-null    int64         
 3   periodo                                 114 non-null    category      
 4   parados_total                           114 non-null    float64       
 5   diferencia_parados_total                114 non-null    float64       
 6   parados_menos_de_25                     114 non-null    float64       
 7   diferencia_parados_menos_de_25          114 non-null    float64       
 8   parados_25_o_mas                        114 non-null    float64       
 9   diferencia_parados_25_o

### Pivotando la tabla moderna

In [97]:
df_02_25 = df_02_25.pivot_table(
    index=['fecha', 'año', 'trimestre', 'periodo'],
    columns=['Sexo', 'Edad'],
    values='Total',
    observed=True
).reset_index()


# Convirtiendo el nombre de las columnas en snake_case
df_02_25.columns = ['_'.join(columna).strip('_') if isinstance(columna, tuple) else columna for columna in df_02_25.columns.values]
df_02_25.columns = [columna.lower().replace(' ', '_') for columna in df_02_25.columns]

df_02_25 = df_02_25.sort_values('fecha').reset_index(drop=True)
df_02_25.head()

,fecha,año,trimestre,periodo,ambos_sexos_25_o_mas,ambos_sexos_menos_de_25,ambos_sexos_total,hombres_25_o_mas,hombres_menos_de_25,hombres_total,mujeres_25_o_mas,mujeres_menos_de_25,mujeres_total
0,2002-01-01,2002,1,2002Q1,1611.9,540.9,2152.8,678.9,257.0,936.1,933.0,283.9,1216.7
1,2002-04-01,2002,2,2002Q2,1577.0,526.3,2103.3,653.0,234.0,887.1,924.0,292.3,1216.2
2,2002-07-01,2002,3,2002Q3,1636.4,559.5,2196.0,683.4,269.9,953.3,953.1,289.6,1242.7
3,2002-10-01,2002,4,2002Q4,1664.8,567.7,2232.4,711.1,280.9,992.0,953.7,286.9,1240.5
4,2003-01-01,2003,1,2003Q1,1762.7,565.8,2328.5,734.0,278.0,1011.9,1028.9,287.8,1316.6


### Explorando la unión y las posibles columnas iteresantes

In [98]:
# Visualización: zona de solape de metodologías del paro, 2001-2006
df_total = df_76_04[['fecha', 'parados_total', 'parados_total_antigua']]
df_total2 = df_02_25[['fecha', 'ambos_sexos_total']]
df_transicion= pd.concat([df_total, df_total2], ignore_index=True)

df_transicion = df_transicion[
    (df_transicion['fecha'] >= '2001-01-01') & (df_transicion['fecha'] <= '2006-01-01')
]

px.line(
    df_transicion,
    x='fecha',
    y=['parados_total', 'parados_total_antigua','ambos_sexos_total'],
    title='Zona de cambio de metodologías y años posteriores',
    labels={'value': 'Total de parados', 'fecha': 'Fecha'},
    markers=True
)

In [99]:
#Dropeando la información de la encuesta que ya se encuentra en las versiones actualizadas
df_76_04 = df_76_04[df_76_04['fecha'] <= '2002-01-01']

In [100]:
df_76_04.columns

Index(['fecha', 'año', 'trimestre', 'periodo', 'parados_total',
       'diferencia_parados_total', 'parados_menos_de_25',
       'diferencia_parados_menos_de_25', 'parados_25_o_mas',
       'diferencia_parados_25_o_mas', 'parados_total_antigua',
       'parados_menos_de_25_antigua', 'parados_25_o_mas_antigua',
       'parados_varones_total', 'diferencia_parados_varones_total',
       'parados_varones_menos_de_25', 'diferencia_parados_varones_menos_de_25',
       'parados_varones_25_o_mas', 'diferencia_parados_varones_25_o_mas',
       'parados_varones_total_antigua', 'parados_varones_menos_de_25_antigua',
       'parados_varones_25_o_mas_antigua', 'parados_mujeres_total',
       'diferencia_parados_mujeres_total', 'parados_mujeres_menos_de_25',
       'diferencia_parados_mujeres_menos_de_25', 'parados_mujeres_25_o_mas',
       'diferencia_parados_mujeres_25_o_mas', 'parados_mujeres_total_antigua',
       'parados_mujeres_menos_de_25_antigua',
       'parados_mujeres_25_o_mas_antigua'],

In [101]:
df_02_25.columns

Index(['fecha', 'año', 'trimestre', 'periodo', 'ambos_sexos_25_o_mas',
       'ambos_sexos_menos_de_25', 'ambos_sexos_total', 'hombres_25_o_mas',
       'hombres_menos_de_25', 'hombres_total', 'mujeres_25_o_mas',
       'mujeres_menos_de_25', 'mujeres_total'],
      dtype='object')

### Cambios de nombre a columnas

In [102]:
columnas_antiguas = [columna for columna in df_76_04.columns if columna.endswith('_antigua')]
df_76_04 = df_76_04.drop(columns=columnas_antiguas)

df_76_04 = df_76_04.rename(columns={
    # Ambos sexos
    'parados_total': 'parados_ambos_sexos_total',
    'diferencia_parados_total': 'diferencia_parados_ambos_sexos_total',
    'parados_menos_de_25': 'parados_ambos_sexos_menos_de_25',
    'diferencia_parados_menos_de_25': 'diferencia_parados_ambos_sexos_menos_de_25',
    'parados_25_o_mas': 'parados_ambos_sexos_25_o_mas',
    'diferencia_parados_25_o_mas': 'diferencia_parados_ambos_sexos_25_o_mas',

    # Hombres
    'parados_varones_total': 'parados_hombres_total',
    'diferencia_parados_varones_total': 'diferencia_parados_hombres_total',
    'parados_varones_menos_de_25': 'parados_hombres_menos_de_25',
    'diferencia_parados_varones_menos_de_25': 'diferencia_parados_hombres_menos_de_25',
    'parados_varones_25_o_mas': 'parados_hombres_25_o_mas',
    'diferencia_parados_varones_25_o_mas': 'diferencia_parados_hombres_25_o_mas',

    # Mujeres
    'parados_mujeres_total': 'parados_mujeres_total',
    'diferencia_parados_mujeres_total': 'diferencia_parados_mujeres_total',
    'parados_mujeres_menos_de_25': 'parados_mujeres_menos_de_25',
    'diferencia_parados_mujeres_menos_de_25': 'diferencia_parados_mujeres_menos_de_25',
    'parados_mujeres_25_o_mas': 'parados_mujeres_25_o_mas',
    'diferencia_parados_mujeres_25_o_mas': 'diferencia_parados_mujeres_25_o_mas'
})

df_02_25 = df_02_25.rename(columns={
    # Ambos sexos
    'ambos_sexos_total': 'parados_ambos_sexos_total',
    'ambos_sexos_menos_de_25': 'parados_ambos_sexos_menos_de_25',
    'ambos_sexos_25_o_mas': 'parados_ambos_sexos_25_o_mas',

    # Hombres
    'hombres_total': 'parados_hombres_total',
    'hombres_menos_de_25': 'parados_hombres_menos_de_25',
    'hombres_25_o_mas': 'parados_hombres_25_o_mas',

    # Mujeres
    'mujeres_total': 'parados_mujeres_total',
    'mujeres_menos_de_25': 'parados_mujeres_menos_de_25',
    'mujeres_25_o_mas': 'parados_mujeres_25_o_mas'
})

# 3. Unión en un único dataset

In [103]:
df_final= pd.concat([df_76_04, df_02_25], ignore_index=True)

df_final = df_final.sort_values('fecha').reset_index(drop=True)
# Eliminar duplicados de ruptura metodológica 2002, conservar serie más moderna
df_final = df_final.drop_duplicates(subset=['fecha'], keep='last').reset_index(drop=True)
print(f'Filas tras deduplicación: {len(df_final)} | duplicados eliminados: {df_final["fecha"].duplicated().sum()}')
df_final

Filas tras deduplicación: 198 | duplicados eliminados: 0


,fecha,año,trimestre,periodo,parados_ambos_sexos_total,diferencia_parados_ambos_sexos_total,parados_ambos_sexos_menos_de_25,diferencia_parados_ambos_sexos_menos_de_25,parados_ambos_sexos_25_o_mas,diferencia_parados_ambos_sexos_25_o_mas,...,parados_hombres_menos_de_25,diferencia_parados_hombres_menos_de_25,parados_hombres_25_o_mas,diferencia_parados_hombres_25_o_mas,parados_mujeres_total,diferencia_parados_mujeres_total,parados_mujeres_menos_de_25,diferencia_parados_mujeres_menos_de_25,parados_mujeres_25_o_mas,diferencia_parados_mujeres_25_o_mas
0,1976-07-01,1976,3,1976Q3,589.000000,0.000000,280.000000,0.000000,309.000000,0.000000,...,151.100000,0.000000,255.800000,0.000000,182.100000,0.000000,129.000000,0.000000,53.200000,0.000000
1,1976-10-01,1976,4,1976Q4,627.771632,-0.228368,303.871632,-0.128368,323.900000,0.000000,...,166.871632,-0.128368,274.343160,-0.056840,186.600000,0.000000,137.000000,0.000000,49.698869,-0.001131
2,1977-01-01,1977,1,1977Q1,652.867832,-1.232168,313.474892,-0.625108,339.392940,-0.507060,...,174.703159,-0.596841,289.401563,-0.498437,188.763110,-0.136890,138.771733,-0.228267,49.991377,-0.008623
3,1977-04-01,1977,2,1977Q2,629.136744,-2.263256,307.356591,-0.943409,321.780153,-1.319847,...,170.481873,-0.618127,274.573929,-1.126071,184.080942,-0.519058,136.874718,-0.425282,47.206224,-0.193776
4,1977-07-01,1977,3,1977Q3,707.184698,-3.715302,373.656013,-1.943987,333.528686,-1.871314,...,202.764550,-0.835450,280.641865,-1.458135,223.778283,-1.421717,170.891463,-1.108537,52.886821,-0.313179
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193,2024-10-01,2024,4,2024Q4,2595.500000,NaN,434.400000,NaN,2161.100000,NaN,...,227.000000,NaN,1005.900000,NaN,1362.600000,NaN,207.400000,NaN,1155.100000,NaN
194,2025-01-01,2025,1,2025Q1,2789.200000,NaN,450.900000,NaN,2338.200000,NaN,...,245.100000,NaN,1065.000000,NaN,1479.100000,NaN,205.800000,NaN,1273.500000,NaN
195,2025-04-01,2025,2,2025Q2,2553.100000,NaN,450.600000,NaN,2102.500000,NaN,...,232.300000,NaN,970.900000,NaN,1350.000000,NaN,218.400000,NaN,1131.800000,NaN
196,2025-07-01,2025,3,2025Q3,2613.200000,NaN,504.100000,NaN,2109.000000,NaN,...,257.900000,NaN,922.500000,NaN,1432.800000,NaN,246.300000,NaN,1186.400000,NaN


In [104]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 198 entries, 0 to 197
Data columns (total 22 columns):
 #   Column                                      Non-Null Count  Dtype         
---  ------                                      --------------  -----         
 0   fecha                                       198 non-null    datetime64[ns]
 1   año                                         198 non-null    int64         
 2   trimestre                                   198 non-null    int64         
 3   periodo                                     198 non-null    string        
 4   parados_ambos_sexos_total                   198 non-null    float64       
 5   diferencia_parados_ambos_sexos_total        102 non-null    float64       
 6   parados_ambos_sexos_menos_de_25             198 non-null    float64       
 7   diferencia_parados_ambos_sexos_menos_de_25  102 non-null    float64       
 8   parados_ambos_sexos_25_o_mas                198 non-null    float64       
 9   diferencia

In [105]:
px.line(df_final, x='fecha', y = ['parados_ambos_sexos_total', 'parados_hombres_total', 'parados_mujeres_total'], title= 'Parados, totales y totales por sexo', subtitle=('Miles de personas'))

In [106]:
px.line(df_final, x='fecha', y = ['parados_ambos_sexos_menos_de_25', 'parados_hombres_menos_de_25', 'parados_mujeres_menos_de_25'], title= 'Parados menores de 25, totales y totales por sexo', subtitle=('Miles de personas')).update_yaxes(range=[0, 1500], dtick=250)

Estacionalidad más reciente: el paro aumenta en el tercer trimestre, reportado en julio.

In [107]:
px.line(df_final, x='fecha', y = ['parados_ambos_sexos_25_o_mas', 'parados_hombres_25_o_mas', 'parados_mujeres_25_o_mas'], title= 'Parados de 25 años o más, totales y totales por sexo', subtitle=('Miles de personas')).update_yaxes(range=[0, 6000])

Estacionalidad más reciente: el paro aumenta en el primer trimestre, reportado en enero.

# 4. Guardando el dataset final en un archivo

In [108]:
# ── Recortar al rango de tasa de paro y guardar en Datasets ────────────
from pathlib import Path
FECHA_INICIO = '1976-07-01'
FECHA_FIN    = '2025-10-01'
ruta_datasets = Path(r'C:\Users\marco\PycharmProjects\TFM_Marcos\Datasets')

df_final = df_final[(df_final['fecha'] >= FECHA_INICIO) & (df_final['fecha'] <= FECHA_FIN)].copy()
df_final.to_csv(ruta_datasets / 'parados_sexo_edad.csv',
                index=False,
                encoding='utf-8-sig')

# 5. Información acerca del dataset final

In [109]:
df_final.attrs['nombre'] = 'df_final'
print('Información acerca del dataset final:')
print(f'{df_final.shape[0]} trimestres × {df_final.shape[1]} columnas | Rango: {df_final["fecha"].min().date()} → {df_final["fecha"].max().date()}')
columnas = df_final.columns.tolist()
print(f'  Columnas: {columnas}')
nulos = df_final.isna().sum()
nulos_col = nulos[nulos > 0]
if len(nulos_col) > 0:
    print(f'ALERTA  {df_final.attrs['nombre']}: nulos en {nulos_col.to_dict()}')
else:
    print(f'CORRECTO  {df_final.attrs['nombre']}: sin valores nulos')
print()

Información acerca del dataset final:
198 trimestres × 22 columnas | Rango: 1976-07-01 → 2025-10-01
  Columnas: ['fecha', 'año', 'trimestre', 'periodo', 'parados_ambos_sexos_total', 'diferencia_parados_ambos_sexos_total', 'parados_ambos_sexos_menos_de_25', 'diferencia_parados_ambos_sexos_menos_de_25', 'parados_ambos_sexos_25_o_mas', 'diferencia_parados_ambos_sexos_25_o_mas', 'parados_hombres_total', 'diferencia_parados_hombres_total', 'parados_hombres_menos_de_25', 'diferencia_parados_hombres_menos_de_25', 'parados_hombres_25_o_mas', 'diferencia_parados_hombres_25_o_mas', 'parados_mujeres_total', 'diferencia_parados_mujeres_total', 'parados_mujeres_menos_de_25', 'diferencia_parados_mujeres_menos_de_25', 'parados_mujeres_25_o_mas', 'diferencia_parados_mujeres_25_o_mas']
ALERTA  df_final: nulos en {'diferencia_parados_ambos_sexos_total': 96, 'diferencia_parados_ambos_sexos_menos_de_25': 96, 'diferencia_parados_ambos_sexos_25_o_mas': 96, 'diferencia_parados_hombres_total': 96, 'diferencia

Los nulos en las diferencias se explican a que solo es calculada en referencia a la metodología antigua. Anterior a 2002, de 2002 en adelante todos los datos para esas columnas son nulos.

In [110]:
df_final.describe()

,fecha,año,trimestre,parados_ambos_sexos_total,diferencia_parados_ambos_sexos_total,parados_ambos_sexos_menos_de_25,diferencia_parados_ambos_sexos_menos_de_25,parados_ambos_sexos_25_o_mas,diferencia_parados_ambos_sexos_25_o_mas,parados_hombres_total,...,parados_hombres_menos_de_25,diferencia_parados_hombres_menos_de_25,parados_hombres_25_o_mas,diferencia_parados_hombres_25_o_mas,parados_mujeres_total,diferencia_parados_mujeres_total,parados_mujeres_menos_de_25,diferencia_parados_mujeres_menos_de_25,parados_mujeres_25_o_mas,diferencia_parados_mujeres_25_o_mas
count,198,198.000000,198.000000,198.000000,102.000000,198.000000,102.000000,198.000000,102.000000,198.000000,...,198.000000,102.000000,198.000000,102.000000,198.000000,102.000000,198.000000,102.000000,198.000000,102.000000
mean,2001-02-14 12:07:16.363636352,2000.747475,2.510101,2913.266867,-231.838694,770.351003,-68.274817,2142.915864,-163.578583,1501.967412,...,393.111920,-31.848248,1108.848926,-72.591359,1411.300179,-127.402586,377.241104,-36.414805,1034.070685,-90.993675
min,1976-07-01 00:00:00,1976.000000,1.000000,589.000000,-500.117120,280.000000,-118.860587,309.000000,-385.808940,406.900000,...,151.100000,-59.975396,255.800000,-150.819518,182.100000,-320.912160,129.000000,-69.513170,47.206224,-251.398990
25%,1988-10-24 00:00:00,1988.250000,2.000000,2202.411775,-413.404934,525.050000,-102.566861,1431.060393,-310.238328,1012.525000,...,265.150000,-45.143446,707.575000,-122.546379,1053.250000,-244.944855,254.754598,-56.276289,674.270866,-181.955796
50%,2001-02-15 00:00:00,2001.000000,3.000000,2757.254217,-203.666352,766.950000,-74.657017,1741.200000,-128.339480,1403.800000,...,376.481670,-34.250443,991.166868,-62.857097,1311.262731,-110.237095,358.100000,-41.651108,961.100000,-67.339621
75%,2013-06-08 06:00:00,2013.000000,3.750000,3357.444818,-77.147430,966.175000,-37.686439,2727.750000,-39.462546,1734.667780,...,494.625000,-20.277514,1268.850000,-28.744351,1706.888755,-28.099620,496.814521,-17.206425,1460.150000,-10.868195
max,2025-10-01 00:00:00,2025.000000,4.000000,6278.200000,0.000000,1355.030481,0.000000,5297.600000,0.000000,3358.400000,...,739.078664,0.000000,2815.600000,0.000000,2919.900000,0.000000,672.906799,0.000000,2481.900000,0.000000
std,NaN,14.327102,1.120822,1210.589668,165.770290,277.537546,37.551484,1165.063029,130.561240,644.952871,...,147.223560,16.725640,582.655234,49.060024,611.435377,102.929358,138.717914,21.184522,618.497769,83.362402


In [111]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 198 entries, 0 to 197
Data columns (total 22 columns):
 #   Column                                      Non-Null Count  Dtype         
---  ------                                      --------------  -----         
 0   fecha                                       198 non-null    datetime64[ns]
 1   año                                         198 non-null    int64         
 2   trimestre                                   198 non-null    int64         
 3   periodo                                     198 non-null    string        
 4   parados_ambos_sexos_total                   198 non-null    float64       
 5   diferencia_parados_ambos_sexos_total        102 non-null    float64       
 6   parados_ambos_sexos_menos_de_25             198 non-null    float64       
 7   diferencia_parados_ambos_sexos_menos_de_25  102 non-null    float64       
 8   parados_ambos_sexos_25_o_mas                198 non-null    float64       
 9   diferencia